In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np

batch_size = 32

X = np.load("input.npy")   # shape: (N, 20, 11)
y = np.load("label.npy")   # shape: (N,)

classes = ['normal', 'dos', 'fuzzing', 'spoofing']
# label: 0, 1, 2, 3

class CANDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test
    random_state=42,
    shuffle=True,
    stratify=y 
)
    

train_dataset = CANDataset(X_train, y_train)
test_dataset  = CANDataset(X_test, y_test)

trainloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

testloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self, num_classes = 4):
        super().__init__()

       # input: (batch, 11, 20)
        self.conv1 = nn.Conv1d(
            in_channels=11,
            out_channels=256,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=512,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool1d(kernel_size=2)

        # 20 → 10 → 5
        self.fc1 = nn.Linear(512 * 5, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # x: (batch, 20, 11)
        x = x.permute(0, 2, 1)  # → (batch, 11, 20)

        x = self.pool(F.relu(self.conv1(x)))  # (batch, 32, 10)
        x = self.pool(F.relu(self.conv2(x)))  # (batch, 64, 5)

        x = torch.flatten(x, 1)               # (batch, 64*5)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                       # (batch, 4)

        return x
    
net = Net()

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001, momentum=0.9)

num_epochs = 10

for epoch in range(num_epochs):

    net.train()
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)                 # (batch, 4)
        loss = criterion(outputs, labels)     # labels ∈ {0,1,2,3}

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 200 == 0:
            print(f"[Epoch {epoch+1}, Iter {i+1}] loss: {running_loss / 200:.4f}")
            running_loss = 0.0

print("Finished Training")

In [ ]:
net.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)                 # (batch, 4)
        _, predicted = torch.max(outputs, 1) # class index

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []

net.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print(classification_report(
    y_true, y_pred,
    target_names=["normal", "dos", "fuzzing", "spoofing"]
))

print(confusion_matrix(y_true, y_pred))